In [1]:
import os
import sys

current_path = os.path.abspath(os.getcwd())
base_path = os.path.abspath(os.path.join(current_path, '../../../'))
sys.path.append(base_path)

import numpy as np
import matplotlib.pyplot as plt  
import pandas as pd
from tqdm import tqdm
import pandas as pd
from IPython.display import display


import project_images_loader as pil
import utils.util as util
import utils.functions as fc
import utils.nifti_functions as nfc

import utils.util_freesurfer_segmentation as util_freesurfer
from PIL import Image
import random
import itk

np.set_printoptions(precision=4)

import test_code.creator_2D_images.constants as cts

import scipy.ndimage
import SimpleITK as sitk


SERVER: False


In [2]:
matching_ids = pil.get_matching_ids()

# Compute inverse parameter files for HUNT3 test data

In [3]:
def find_inverse_registration_param_files(hunt, id):
    if hunt == 3:
        path_name_fixed = pil.get_T1_MNI_template_path_name(prep=True, absolute=True)
    elif hunt == 4:
        path_name_fixed = cts.get_registered_image_path_name(3, id)

    # step 1: compute the inverse parameters
    fixed_image = itk.imread(path_name_fixed, itk.F)

    path_name_pram_to_inver_file = cts.get_registered_pfile_path_name(hunt, id)
    path_name_pram_file_affine = pil.get_parameters_file_path_name(38, pil.PARAMETERS_FILE_AFFINE)
    param_obj = itk.ParameterObject.New()
    # param_obj.AddParameterFile(path_name_pram_to_inver_file)
    param_obj.AddParameterFile(path_name_pram_file_affine)

    res_similarity, transform_parameters = itk.elastix_registration_method(fixed_image, fixed_image,
                                                                            parameter_object=param_obj,
                                                                            initial_transform_parameter_file_name=path_name_pram_to_inver_file,
                                                                            log_to_console=True)
    transform_parameters.SetParameter('InitialTransformParameterFileName','NoInitialTransform')
    param_reg_path_name = cts.get_INV_registered_pfile_path_name(hunt, id)
    transform_parameters.WriteParameterFile(transform_parameters, param_reg_path_name)

def compute_all_inverse_param_files(ids):
    for id in tqdm(ids):
        for hunt in [3]:
            find_inverse_registration_param_files(hunt, id)

val_images = pil.get_matching_ids()[124:621] # 
compute_all_inverse_param_files(val_images)


100%|██████████| 497/497 [48:34<00:00,  5.86s/it]


In [3]:


def save_image_back(hunt, id):
    param_obj = itk.ParameterObject.New()
    param_reg_path_name = cts.get_INV_registered_pfile_path_name(hunt, id)
    
    param_obj.AddParameterFile(param_reg_path_name)

    moving_image_path_name = cts.get_registered_image_path_name(hunt, id)
    img_moving = itk.imread(moving_image_path_name, itk.F)
    reg_img_moving = itk.transformix_filter(img_moving, param_obj)

    img_reg_path_name = f"{current_path}/images/{id}_{hunt-3}_REG_BACK.nii.gz"
    itk.imwrite(reg_img_moving, img_reg_path_name)

id = matching_ids[366]
hunt = 3
find_inverse_registration_param_files(hunt, id)
# save_image_back(hunt, id)



In [6]:

#(InitialTransformParameterFileName "C:/Users/Agustin/Master/Master/S4/code/test_code/creator_2D_images/images_3D_preprocessed/HUNT3/00039/00039_0_PREP_MNI_DFIELD_AFF.txt")
